# TC_02: XGBoost clean baseline
Full diagnostic pipeline on clean data using XGB model.
UQ: Deep Ensemble,
XAI: SHAP + LIME

In [1]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt

sys.path.append('..')
os.chdir('..')
plt.style.use('default')
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'

from src.utils.config_loader import load_config
from src.utils.data_loader import load_tabular_data
from src.data_diagnostics.quality_checks import check_tabular_quality
from src.utils.visualization import plot_class_distribution, plot_correlation, plot_feature_boxplots
from src.utils.performance_tracker import PerformanceTracker
from src.models.tabular_models import train_xgboost, evaluate_model_rf_xgb
from src.inference_diagnostics.uncertainty import deep_ensemble_prediction_sklern
from src.inference_diagnostics.explainability import shap_tab,lime_tab,calculate_consistency_tabular
from src.inference_diagnostics.flagging import assign_flags,evaluate_flags
from src.utils.visualization import plot_accuracy_comparison, plot_uncertainty_distribution, plot_flag_distribution
from src.utils.report_generator import generate_report,save_report
from src.utils.config_loader import load_config, get_tabular_config, save_threshold
from src.utils.per_sample_logger import save_per_sample, build_experiment_id

config = load_config()
tracker = PerformanceTracker()

In [2]:
# Identify this run
dataset_short = get_tabular_config(config)['short_name']
model_name = 'xgb'
test_case = 'TC02'
experiment_id = build_experiment_id(dataset_short, model_name, test_case)
print(f"Experiment: {experiment_id}")

Experiment: adult_xgb_TC02


### Load data and EDA

In [3]:
X_train, X_test, y_train, y_test = load_tabular_data(config)
report = check_tabular_quality(X_train, y_train, config)
plot_class_distribution(report, f"RF {get_tabular_config(config)['name']} Baseline", config)
plot_correlation(report, f"RF {get_tabular_config(config)['name']} Baseline", config)
plot_feature_boxplots(X_train, f"RF {get_tabular_config(config)['name']} Baseline", config)

Loading dataset.
Dataset: UCI Adult Income
Duplicate rows kept (6374 found).
Found 6465 missing values. Filling with mode.
Data loaded for 39073 train, 9769 test
 Features: 12
EDA started for tabular data.
Samples: 39073, Features: 12
Class distribution: {0: 29724, 1: 9349}
Imbalance ratio: 0.3145
Duplicate rows: 5422
Total outlier count: 5915
EDA completed for UCI Adult Income
Saved: results/figures\class_distribution_rf_uci_adult_income_baseline.png
Saved: results/figures\correlation_rf_uci_adult_income_baseline.png
Saved: results/figures\boxplot_rf_uci_adult_income_baseline.png


### Train XGBoost baseline

In [4]:
tracker.start_performance_track()
xgb_model = train_xgboost(X_train, y_train, config)
tracker.stop_performance_track("XGBoost training")

tracker.start_performance_track()
xgb_accuracy, xgb_prediction, xgb_report = evaluate_model_rf_xgb(xgb_model, X_test, y_test)
tracker.stop_performance_track("XGBoost evaluation")

XGBoost training started.
XGBoost training completed.
XGBoost training: Time:0.20s, Memory:133.32MB
Model evaluation started for RF,XGB
{'0': {'precision': 0.8971450420596482, 'recall': 0.9472480150719957, 'f1-score': 0.9215160044511357, 'support': 7431.0}, '1': {'precision': 0.7961518460738429, 'recall': 0.6548331907613345, 'f1-score': 0.7186106547758742, 'support': 2338.0}, 'accuracy': 0.8772648172791483, 'macro avg': {'precision': 0.8466484440667456, 'recall': 0.8010406029166651, 'f1-score': 0.8200633296135049, 'support': 9769.0}, 'weighted avg': {'precision': 0.8729744931585516, 'recall': 0.8772648172791483, 'f1-score': 0.8729549738911232, 'support': 9769.0}}
XGBoost evaluation: Time:0.01s, Memory:0.19MB


{'operation': 'XGBoost evaluation', 'time_seconds': 0.01, 'memory_usage': 0.19}

### Uncertainty Quantification (Deep Ensembles)

In [5]:
tracker.start_performance_track()
de_means_probabilities, de_uncertainty = deep_ensemble_prediction_sklern(train_xgboost, X_train, y_train, X_test, config)
tracker.stop_performance_track("XGB Deep Ensemble prediction")

de_threshold = round(float(np.percentile(de_uncertainty, 90)), 4)
save_threshold(config, dataset_short, model_name, 'de', de_threshold)
de_y_prediction = de_means_probabilities.argmax(axis=1)
print(f"Deep Ensembl uncertainty status:")
print(f"Mean: {de_uncertainty.mean()}")
print(f"Max: {de_uncertainty.max()}")
print(f" 90th percentile (threshold): {de_threshold}")

plot_uncertainty_distribution(de_uncertainty, "XGB Deep Ensemble", de_threshold, config)

Deep Ensemble started for tabular and sklern.
Training model with seed 0
XGBoost training started.
XGBoost training completed.
Training model with seed 1
XGBoost training started.
XGBoost training completed.
Training model with seed 2
XGBoost training started.
XGBoost training completed.
Training model with seed 3
XGBoost training started.
XGBoost training completed.
Training model with seed 4
XGBoost training started.
XGBoost training completed.
Deep Ensemble finished for tabular sklern.
XGB Deep Ensemble prediction: Time:0.37s, Memory:7.41MB
Threshold saved: adult_xgb_de = 0.0
Deep Ensembl uncertainty status:
Mean: 8.290790276532789e-09
Max: 3.3527612686157227e-08
 90th percentile (threshold): 0.0
Saved: results/figures\uncertainty_distribution_xgb_deep_ensemble.png


### Explainability (SHAP and LIME)

In [6]:
# SHAP
tracker.start_performance_track()
shap_values, shap_samples = shap_tab(xgb_model, X_train, X_test, config, is_pytorch = False)
tracker.stop_performance_track("XGB SHAP")
print(f"SHAP values shape: {shap_values.shape}")

SHAP started.


  0%|          | 0/200 [00:00<?, ?it/s]

SHAP finished. Explained: 200 samples.
XGB SHAP: Time:7.32s, Memory:0.54MB
SHAP values shape: (200, 12, 2)


In [7]:
# LIME
tracker.start_performance_track()
lime_explanations, lime_samples = lime_tab(xgb_model, X_train, X_test, config, is_pytorch = False)
tracker.stop_performance_track("XGB LIME")
print(f"LIME explanations: {len(lime_explanations)}")

LIME started.
Explained 50 / 200 samples.
Explained 100 / 200 samples.
Explained 150 / 200 samples.
Explained 200 / 200 samples.
LIME finished. Explained: 200 samples.
XGB LIME: Time:0.98s, Memory:4.46MB
LIME explanations: 200


In [8]:
# Calculate consistency
feature_names = list(X_train.columns)
consistency_scores = calculate_consistency_tabular(shap_values, lime_explanations, feature_names, top=5)
print(f"Mean consistency: {consistency_scores.mean()}")
print(f"Min consistency: {consistency_scores.min()}")
print(f"Max consistency: {consistency_scores.max()}")


Calculate consistency tabular.
Mean consistency: 0.502
Min consistency: 0.2
Max consistency: 0.8


### Flagging

In [9]:
# Real uncertainty value for XGB is near 0 and meaning less ofr flagging system.
xgb_uq_threshold = float(de_uncertainty.max()) + 1.0
xgb_flags = assign_flags(de_uncertainty[:len(consistency_scores)], consistency_scores, xgb_uq_threshold, 0.5)

print("XGB flagging results:")
xgb_flagging_result = evaluate_flags(xgb_flags, xgb_prediction[:len(consistency_scores)], y_test[:len(consistency_scores)])

plot_flag_distribution(xgb_flags, 'XGB Baseline', config)

XGB flagging results:
RED: Count: 0 Accuracy:0.0%
YELLOW: Count: 100 Accuracy:92.0%
GREEN: Count: 100 Accuracy:88.0%
Flagging system is working
Saved: results/figures\flagging_distribution_xgb_baseline.png


In [10]:
# Per sample results for the ablation and metrics analysis
save_per_sample(
    config,
    experiment_id,
    y_true=y_test,
    y_pred=xgb_prediction,
    mc_uncertainty=None,
    de_uncertainty=de_uncertainty,
    consistency=consistency_scores,
)

Per sample results saved: results/per_sample\adult_xgb_TC02.csv (200 rows)


,sample_id,y_true,y_pred,correct,mc_uncertainty,de_uncertainty,consistency
0,0,0,0,1,NaN,2.328306e-10,0.2
1,1,0,0,1,NaN,0.000000e+00,0.4
2,2,0,0,1,NaN,0.000000e+00,0.6
3,3,0,0,1,NaN,0.000000e+00,0.4
4,4,1,1,1,NaN,1.490116e-08,0.6
...,...,...,...,...,...,...,...
195,195,0,0,1,NaN,2.980232e-08,0.6
196,196,0,0,1,NaN,0.000000e+00,0.4
197,197,0,0,1,NaN,0.000000e+00,0.6
198,198,1,1,1,NaN,0.000000e+00,0.4


### Save baseline report


In [11]:
xgb_baseline = {
    'model': 'XGBoost',
    'accuracy': round(xgb_accuracy, 4),
    'classification_report': xgb_report,
    'de_uncertainty': {
        'mean': round(float(de_uncertainty.mean()), 8),
        'max': round(float(de_uncertainty.max()), 8),
        'threshold': de_threshold,
        'findings': 'Near zero uncertainty and XGB internal ensemble makes external DE redundant'
    },
    'consistency':{
        'mean': round(float(consistency_scores.mean()), 4),
        'min': round(float(consistency_scores.min()), 4),
        'max': round(float(consistency_scores.max()), 4)
    },
    'flagging': xgb_flagging_result,
    'performance': tracker.get_results()
}

report_output = generate_report(config, f"{get_tabular_config(config)['name']} - XGB clean baseline", stage1_result=xgb_baseline)
save_report(report_output, f'{dataset_short.upper()}_TC_02_XGBoost_Clean_Baseline.json', config)

Generating report.
Diagnostic report generated.
Saving report.
